[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/42.%20The%20Peak%20Energy%20Consumption%20%26%20Load%20Shifting%20Problem/P42-Tier-4_executed.ipynb)

---


# 42. The Peak Energy Consumption & Load Shifting Problem

## Tier 4: Deep Q-Networks for Dynamic Energy Management

### Goal

Build a small, self-contained energy-management simulation and train a Deep Q-Network (DQN) agent to choose hourly operation-start and energy-storage actions. The agent learns from simulated experience; it does not receive an optimal schedule in advance. A random policy provides a simple reference for interpreting the evaluation results.

### Inputs and outputs

The environment uses a 24-hour demand profile, hourly electricity prices, a set of operations with durations and energy requirements, and a storage system. Training produces a neural network that estimates the value of each available action in the current observed state. Evaluation reports episode reward, energy cost, and peak demand for the trained policy and for the random-policy reference.

### Scope and modeling assumptions

- Each simulation step represents one hour. Base demand is measured in MW; operation and storage power are measured in kW. Electricity prices are in dollars per kWh.
- An operation's energy value is its total energy requirement in kWh. While it runs, that energy is spread evenly over its stated duration. An operation started at hour \(t\) contributes load from the following hour onward.
- The storage rate is in kW and its state of charge is in kWh. The stated efficiency is applied to both charging and discharging, so it is a one-way efficiency; round-trip efficiency is its square.
- The observation contains the current hour, current and near-term prices, current demand, peak and cost accumulated so far, storage information, and selected operation statuses. Running-operation features include remaining duration. Prices beyond the forecast window are not observed, so this compact observation does not make every randomized episode fully observable.
- Weather, equipment failures, power-quality limits, and operational deadlines are not modeled. Only the eight listed flexible operations can be started; non-flexible operations are not decision variables in this simplified environment.
- Starting an operation is allowed only when its modeled run can finish by the end of the 24-hour horizon. The environment does not include a terminal value for unused stored energy.

### Learning approach

At each hour, the agent observes a state, selects one of 50 discrete actions, and receives a reward. DQN uses a neural network to estimate action values, experience replay to reuse sampled transitions, and a target network to stabilize the bootstrap target. An epsilon-greedy policy sometimes selects a random action during training and otherwise selects an action with the largest estimated value.

The training run uses 2,000 episodes. These are simulation settings, not a promised performance result. Learning curves and evaluation statistics must be generated by running the notebook; no numerical outcome is assumed here.

## 1. State, actions, and reward

### State: the information available to the agent

A state is a numerical summary of the environment at a decision point. It gives the network a consistent input even though the underlying system includes time, prices, storage, and operation status. The observation has 25 components:

| Feature group | Components | Meaning |
|---|---:|---|
| Time | 5 | Normalized hour and episode progress, peak-hour indicator, and sine/cosine time encoding |
| Prices | 6 | Current price and prices for the next five hours, normalized by 0.20 dollars per kWh |
| Demand and accumulated performance | 3 | Current base demand, peak demand so far, and cost so far, each normalized |
| Storage | 3 | State of charge, charge/discharge rate, and efficiency, each normalized |
| Flexible-operation status | 8 | One value for each flexible operation; the value distinguishes idle, running with remaining duration, and completed |

The feature groups have \(5+6+3+3+8=25\) components. This fixed-length vector is the `np.ndarray` returned by `RLEnvironment.get_state`. Normalization keeps many inputs on comparable scales; it does not guarantee that every feature is between zero and one.

### Actions: the choices available at each hour

The discrete action space has 50 entries. Actions 0–9 request the start of one flexible operation. Actions 10–19 request storage charging or discharging at one of five power levels. Actions 20–49 encode combinations of a start request and a storage request. Requests that are unavailable in the current state are treated as no-ops for that part of the action. Storage power is limited by the system's rate and current state of charge; operation starts are accepted only when the operation is idle and can finish within the horizon.

An action is an instruction for the current decision point, not a guarantee that every requested change is feasible. `RLEnvironment.decode_action` translates an integer into named requests, and `RLEnvironment.step` applies feasible requests and advances the simulation by one hour.

### Reward: what the agent is trained to maximize

Let \(D_t\) be demand during hour \(t\), in MW, including base demand, running operations, and the signed storage power change. Let \(p_t\) be the electricity price in dollars per kWh. Let \(P_t=\max_{1\leq j\leq t}D_j\) be the highest demand seen through hour \(t\), and let \(u_t\) be storage power change in kW, positive for charging and negative for discharging. The hourly energy cost and reward are

$$
C_t = 1000 p_t D_t,
\qquad
r_t = -C_t - 0.1(1000P_t)
      + 50\,\mathbf{1}\{6\leq t\leq18,\ u_t<0\}.
$$

At the end of the episode, the environment also applies a demand charge and subtracts it from the final reward:

$$
C_{\mathrm{demand}}=18(1000P_{24}).
$$

Here, multiplying MW by 1,000 converts demand to kW; multiplying that value by the hourly price gives the hourly energy cost. The indicator  mathbf{1}  is one when its condition is true and zero otherwise. The extra hourly peak term and the peak-hour discharge bonus are reward-shaping terms. Consequently, maximizing cumulative reward is not exactly the same as minimizing the reported episode cost, which consists of energy cost plus the demand charge.

## 2. DQN and the learning equations

### Intuition and role

The action-value \(Q(s,a)\) estimates the expected discounted reward from taking action \(a\) in state \(s\), then following the policy. A neural network is useful here because it can estimate values for many state-action combinations without storing a separate table entry for every possible state. Its output has one value per discrete action.

For the transition \((s,a,r,s',d)\), where \(d=1\) at the end of an episode and \(d=0\) otherwise, DQN forms the target

$$
y = r + \gamma(1-d)\max_{a'} Q_{\mathrm{target}}(s',a').
$$

The bootstrap term is zero at a terminal transition. For a nonterminal transition, it uses the largest next-state estimate from the target network. The online network is trained to bring its estimate for the selected action toward \(y\); the other action outputs are kept at their current estimates for that update. With \(A\) actions, the implemented squared-error loss is

$$
L = \frac{1}{A}\sum_{k=1}^{A}\bigl(Q_{\mathrm{online}}(s,k)-q_k^{\mathrm{target}}\bigr)^2,
$$

where \(q_a^{\mathrm{target}}=y\) for the selected action and \(q_k^{\mathrm{target}}=Q_{\mathrm{online}}(s,k)\) for every other action. In the code, `DQN.train_step` constructs this target vector and `DQN._compute_gradients` calculates its gradients.

### Replay, target network, and exploration

Experience replay stores transitions and samples them in a shuffled order, reducing the dependence of each update on the immediately preceding transition. The target network supplies the next-state values in the equation above. It is synchronized with the online network initially and periodically during training. Epsilon-greedy exploration selects a random action with probability `epsilon`; otherwise, it selects the action with the highest predicted value. Epsilon decays during training toward a nonzero minimum.

A DQN state is not the same thing as a neural-network layer: it is the environment's observation at one decision point. The replay buffer stores transitions between such observations. The target equation is the Bellman one-step bootstrap used by this DQN training procedure; it is not a guarantee that the learned policy is globally optimal.

### Network and computational scale

The network has three hidden layers with 128, 64, and 32 units, followed by 50 output units. Each hidden layer uses the rectified linear unit, \(\operatorname{ReLU}(x)=\max(0,x)\), and the output layer is linear. A forward pass computes estimated action values; backpropagation computes how each weight contributed to the loss.

For layer widths \(n_0,n_1,\ldots,n_L\), one forward or backward pass requires work proportional to approximately \(\sum_{\ell=0}^{L-1}n_\ell n_{\ell+1}\). The replay memory grows with its capacity and the size of each stored transition. These costs explain why training takes substantially more computation than choosing one action after training.

In [ ]:
from typing import Any, Dict, List, Tuple
from collections import deque
import random
import time

import matplotlib.pyplot as plt
import numpy as np

# Fixed seeds make the random-number sequences reproducible in a given environment.
np.random.seed(42)
random.seed(42)
plt.style.use("seaborn-v0_8")

## 3. Build the simulation environment

The environment defines the data and transition rules that the agent experiences. `reset` begins a new day, `get_state` constructs its observation, `decode_action` interprets an action, and `step` applies the transition and returns the next observation, reward, terminal flag, and diagnostic information. Keeping these roles visible makes it possible to trace how the learning equations connect to the simulated energy system.

In [ ]:
class RLEnvironment:
    """Hourly energy-management simulation for DQN experiments."""

    def __init__(self):
        self.T = list(range(1, 25))
        self.current_hour = 1

        # Energy is total kWh; duration is the number of modeled operating hours.
        self.operations = {
            "crane_a": {"duration": 4, "energy": 350, "flexible": True},
            "crane_b": {"duration": 4, "energy": 350, "flexible": True},
            "hvac_1": {"duration": 3, "energy": 180, "flexible": True},
            "hvac_2": {"duration": 3, "energy": 180, "flexible": True},
            "ev_charging": {"duration": 6, "energy": 480, "flexible": True},
            "lighting": {"duration": 8, "energy": 120, "flexible": False},
            "refrigeration": {"duration": 8, "energy": 320, "flexible": False},
            "conveyor": {"duration": 5, "energy": 280, "flexible": True},
            "security": {"duration": 12, "energy": 90, "flexible": False},
            "pumping": {"duration": 3, "energy": 240, "flexible": True},
            "ventilation": {"duration": 4, "energy": 190, "flexible": True},
            "gate_ops": {"duration": 2, "energy": 110, "flexible": True},
            "admin": {"duration": 10, "energy": 160, "flexible": True},
            "emergency": {"duration": 1, "energy": 80, "flexible": False},
            "monitoring": {"duration": 24, "energy": 75, "flexible": False},
        }
        self.flexible_ops = [
            "crane_a", "crane_b", "hvac_1", "hvac_2",
            "ev_charging", "conveyor", "pumping", "ventilation",
        ]
        self.action_start_ops = [
            "crane_a", "crane_b", "hvac_1", "hvac_2", "ev_charging",
            "conveyor", "pumping", "ventilation", "gate_ops", "admin",
        ]

        self.base_prices = {
            t: 0.15 if 6 <= t <= 18 else 0.08 for t in self.T
        }
        self.current_prices = self.base_prices.copy()
        self.base_demand = {
            1: 8, 2: 7, 3: 6, 4: 7, 5: 9, 6: 12, 7: 15, 8: 18,
            9: 20, 10: 22, 11: 21, 12: 19, 13: 20, 14: 22, 15: 21,
            16: 19, 17: 17, 18: 15, 19: 13, 20: 11, 21: 10, 22: 9,
            23: 8, 24: 7,
        }

        self.storage = {
            "capacity": 500,       # kWh
            "rate": 100,           # kW
            "efficiency": 0.90,    # one-way efficiency
            "current_soc": 250,    # kWh; initially 50% charged
        }
        self.operation_schedule = {}
        self.operation_status = {op: "idle" for op in self.operations}
        self.episode_step = 0
        self.total_cost = 0.0
        self.peak_demand = 0.0

    def get_state(self) -> np.ndarray:
        """Return the 25-component observation for the current decision point."""
        # A terminal transition advances the clock to 25; hour 24 is used to
        # keep the returned terminal observation within the defined data range.
        hour = min(self.current_hour, 24)
        state = [
            hour / 24.0,
            1.0 if 6 <= hour <= 18 else 0.0,
            self.episode_step / 24.0,
            np.sin(2 * np.pi * hour / 24),
            np.cos(2 * np.pi * hour / 24),
        ]

        state.append(self.current_prices[hour] / 0.20)
        for offset in range(1, 6):
            future_hour = (hour + offset - 1) % 24 + 1
            state.append(self.current_prices[future_hour] / 0.20)

        state.extend([
            self.base_demand[hour] / 25.0,
            self.peak_demand / 30.0 if self.peak_demand > 0 else 0.0,
            self.total_cost / 50000.0 if self.total_cost > 0 else 0.0,
        ])
        state.extend([
            self.storage["current_soc"] / self.storage["capacity"],
            self.storage["rate"] / 200.0,
            self.storage["efficiency"],
        ])

        # Remaining run time helps distinguish running operations with
        # different future load profiles while preserving eight status features.
        for op in self.flexible_ops:
            status = self.operation_status[op]
            if status == "idle":
                status_value = 0.0
            elif status == "completed":
                status_value = 1.5
            else:
                start_hour = self.operation_schedule[op]
                duration = self.operations[op]["duration"]
                remaining = max(0, start_hour + duration - hour + 1)
                status_value = 0.5 + 0.5 * remaining / duration
            state.append(status_value)

        observation = np.asarray(state, dtype=np.float32)
        if observation.shape != (25,):
            raise ValueError(f"Expected a 25-component state, got {observation.shape}.")
        return observation

    def get_action_space_size(self) -> int:
        """Return the number of discrete actions."""
        return 50

    def decode_action(self, action: int) -> Dict[str, Any]:
        """Translate a discrete action into feasible operation and storage requests."""
        if not 0 <= action < self.get_action_space_size():
            raise ValueError("Action must be an integer from 0 through 49.")

        decoded = {}
        hour = min(self.current_hour, 24)

        def add_start_request(op_name: str) -> None:
            duration = self.operations[op_name]["duration"]
            can_finish = hour + duration <= 24
            if self.operation_status[op_name] == "idle" and can_finish:
                decoded["start_operation"] = op_name

        if action < 10:
            add_start_request(self.action_start_ops[action])
        elif action < 20:
            storage_action = action - 10
            if storage_action < 5:
                decoded["storage_charge"] = (storage_action + 1) * 20
            else:
                decoded["storage_discharge"] = (storage_action - 4) * 20
        else:
            combined_index = action - 20
            operation_index = combined_index // 5
            storage_index = combined_index % 5
            combined_ops = ["crane_a", "crane_b", "hvac_1", "hvac_2", "ev_charging"]

            if operation_index < len(combined_ops):
                add_start_request(combined_ops[operation_index])

            if storage_index < 2:
                decoded["storage_charge"] = (storage_index + 1) * 50
            elif storage_index < 4:
                decoded["storage_discharge"] = (storage_index - 1) * 50

        return decoded

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, Dict[str, Any]]:
        """Apply one hourly action and return (next_state, reward, done, info)."""
        if self.current_hour > 24:
            raise RuntimeError("The episode has ended; call reset before stepping again.")

        action_decoded = self.decode_action(action)
        hour = self.current_hour
        current_demand = float(self.base_demand[hour])

        # Operations started by this action begin contributing load next hour.
        for op_name, op_data in self.operations.items():
            if self.operation_status[op_name] == "running":
                energy_rate_kw = op_data["energy"] / op_data["duration"]
                current_demand += energy_rate_kw / 1000.0

        info = {}
        if "start_operation" in action_decoded:
            op_name = action_decoded["start_operation"]
            self.operation_status[op_name] = "running"
            self.operation_schedule[op_name] = hour
            info["action_taken"] = f"Started {op_name}"

        # Power in kW is also energy in kWh over this one-hour time step.
        storage_power_change = 0.0
        efficiency = self.storage["efficiency"]
        if "storage_charge" in action_decoded:
            requested_charge = min(action_decoded["storage_charge"], self.storage["rate"])
            remaining_capacity = self.storage["capacity"] - self.storage["current_soc"]
            grid_energy = min(requested_charge, remaining_capacity / efficiency)
            self.storage["current_soc"] += grid_energy * efficiency
            storage_power_change += grid_energy
            info["storage_charge"] = grid_energy

        if "storage_discharge" in action_decoded:
            requested_discharge = min(action_decoded["storage_discharge"], self.storage["rate"])
            available_output = self.storage["current_soc"] * efficiency
            delivered_energy = min(requested_discharge, available_output)
            self.storage["current_soc"] -= delivered_energy / efficiency
            storage_power_change -= delivered_energy
            info["storage_discharge"] = delivered_energy

        current_demand += storage_power_change / 1000.0
        hour_cost = self.current_prices[hour] * current_demand * 1000.0
        self.total_cost += hour_cost
        self.peak_demand = max(self.peak_demand, current_demand)

        reward = -hour_cost - 0.1 * self.peak_demand * 1000.0
        if 6 <= hour <= 18 and storage_power_change < 0:
            reward += 50.0

        # An operation runs for exactly its stated number of hourly periods.
        for op_name, start_hour in list(self.operation_schedule.items()):
            duration = self.operations[op_name]["duration"]
            if hour >= start_hour + duration:
                self.operation_status[op_name] = "completed"
                del self.operation_schedule[op_name]

        self.current_hour += 1
        self.episode_step += 1
        done = self.current_hour > 24 or self.episode_step >= 24

        if done:
            demand_cost = 18.0 * self.peak_demand * 1000.0
            self.total_cost += demand_cost
            reward -= demand_cost
            info["episode_cost"] = self.total_cost
            info["peak_demand"] = self.peak_demand

        return self.get_state(), reward, done, info

    def reset(self) -> np.ndarray:
        """Start a new episode and return its initial observation."""
        self.current_hour = 1
        self.episode_step = 0
        self.total_cost = 0.0
        self.peak_demand = 0.0
        self.operation_schedule = {}
        self.operation_status = {op: "idle" for op in self.operations}
        self.storage["current_soc"] = 250

        # Each episode has either the base prices or one randomly selected spike.
        self.current_prices = self.base_prices.copy()
        if random.random() < 0.2:
            spike_hour = random.randint(6, 18)
            self.current_prices[spike_hour] *= 1.5

        return self.get_state()


env = RLEnvironment()
print("RL environment initialized")
print(f"State space dimension: {len(env.get_state())}")
print(f"Action space size: {env.get_action_space_size()}")
print(f"Number of operations: {len(env.operations)}")

## 4. Implement the DQN

The online network estimates action values and is updated by gradient descent. The target network is a separate copy used to calculate bootstrap targets. The implementation uses NumPy for the forward pass and backpropagation, so the relationship between the equations and the individual matrix operations remains visible.

In [ ]:
class DQN:
    """Fully connected Q-network and one-step DQN update."""

    def __init__(
        self,
        state_dim: int,
        action_dim: int,
        hidden_dims: List[int] = None,
    ):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.hidden_dims = list(hidden_dims) if hidden_dims is not None else [128, 64, 32]
        self.weights = self._initialize_weights()

        self.learning_rate = 0.001
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995

    def _initialize_weights(self) -> List[np.ndarray]:
        """Initialize layer weights with Xavier uniform scaling."""
        weights = []
        layer_sizes = [self.state_dim] + self.hidden_dims + [self.action_dim]
        for input_size, output_size in zip(layer_sizes[:-1], layer_sizes[1:]):
            limit = np.sqrt(6.0 / (input_size + output_size))
            weights.append(np.random.uniform(-limit, limit, (input_size, output_size)))
            weights.append(np.zeros((1, output_size)))
        return weights

    def _relu(self, values: np.ndarray) -> np.ndarray:
        """Apply the rectified linear unit activation."""
        return np.maximum(0.0, values)

    def _forward(
        self,
        state: np.ndarray,
        weights: List[np.ndarray] = None,
    ) -> np.ndarray:
        """Compute one network output vector."""
        network_weights = self.weights if weights is None else weights
        values = np.asarray(state, dtype=np.float32).reshape(1, -1)
        for index in range(0, len(network_weights) - 2, 2):
            values = self._relu(values @ network_weights[index] + network_weights[index + 1])
        q_values = values @ network_weights[-2] + network_weights[-1]
        return q_values.ravel()

    def predict(self, state: np.ndarray) -> np.ndarray:
        """Return estimated Q-values for every action."""
        return self._forward(state)

    def act(self, state: np.ndarray, training: bool = True) -> int:
        """Select an action using epsilon-greedy exploration when training."""
        if training and random.random() < self.epsilon:
            return random.randrange(self.action_dim)
        return int(np.argmax(self.predict(state)))

    def _compute_gradients(
        self,
        state: np.ndarray,
        target_q: np.ndarray,
    ) -> List[np.ndarray]:
        """Backpropagate the mean squared error through all network layers."""
        values = np.asarray(state, dtype=np.float32).reshape(1, -1)
        activations = [values]
        pre_activations = []
        layer_count = len(self.weights) // 2

        for layer_index in range(layer_count - 1):
            z = activations[-1] @ self.weights[2 * layer_index]
            z += self.weights[2 * layer_index + 1]
            pre_activations.append(z)
            activations.append(self._relu(z))

        q_values = activations[-1] @ self.weights[-2] + self.weights[-1]
        output_error = (2.0 / self.action_dim) * (
            q_values - target_q.reshape(1, -1)
        )
        gradients = [None] * len(self.weights)

        last_layer = layer_count - 1
        gradients[2 * last_layer] = activations[-1].T @ output_error
        gradients[2 * last_layer + 1] = np.sum(output_error, axis=0, keepdims=True)

        error = output_error
        for layer_index in range(layer_count - 2, -1, -1):
            error = error @ self.weights[2 * (layer_index + 1)].T
            error *= (pre_activations[layer_index] > 0).astype(float)
            gradients[2 * layer_index] = activations[layer_index].T @ error
            gradients[2 * layer_index + 1] = np.sum(error, axis=0, keepdims=True)

        return gradients

    def train_step(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
        target_network: "DQN" = None,
    ) -> float:
        """Apply one squared-error Q-learning update and return its loss."""
        current_q = self.predict(state)
        target_q = current_q.copy()
        bootstrap_network = target_network if target_network is not None else self
        next_q = bootstrap_network.predict(next_state)
        target_q[action] = reward + (0.0 if done else self.gamma * np.max(next_q))

        loss = float(np.mean((current_q - target_q) ** 2))
        gradients = self._compute_gradients(state, target_q)
        for index, gradient in enumerate(gradients):
            self.weights[index] -= self.learning_rate * gradient

        if self.epsilon > self.epsilon_min:
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
        return loss

    def update_target_network(self, target_network: "DQN") -> None:
        """Copy online parameters into the target network."""
        target_network.weights = [weight.copy() for weight in self.weights]


state_dim = len(env.get_state())
action_dim = env.get_action_space_size()
dqn_agent = DQN(state_dim, action_dim, hidden_dims=[128, 64, 32])
target_dqn = DQN(state_dim, action_dim, hidden_dims=[128, 64, 32])
dqn_agent.update_target_network(target_dqn)

print("DQN agent created")
print(f"State dimension: {state_dim}")
print(f"Action dimension: {action_dim}")
print(f"Network architecture: {[state_dim] + dqn_agent.hidden_dims + [action_dim]}")

## 5. Train with experience replay

Training repeatedly collects a transition, adds it to replay memory, and updates the online network using sampled transitions. The target network is synchronized at the stated episode interval. The training function returns per-episode metrics so later cells can plot and inspect the same run.

In [ ]:
class ExperienceReplay:
    """Fixed-capacity memory for previously observed transitions."""

    def __init__(self, capacity: int = 10000):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)

    def push(
        self,
        state: np.ndarray,
        action: int,
        reward: float,
        next_state: np.ndarray,
        done: bool,
    ) -> None:
        """Store one transition for later learning."""
        self.buffer.append((state.copy(), action, reward, next_state.copy(), done))

    def sample(self, batch_size: int) -> List[Tuple]:
        """Draw a uniformly sampled batch without replacement."""
        sample_size = min(batch_size, len(self.buffer))
        return random.sample(list(self.buffer), sample_size)

    def __len__(self) -> int:
        return len(self.buffer)


def train_dqn_agent(
    env: RLEnvironment,
    agent: DQN,
    target_agent: DQN,
    num_episodes: int = 2000,
    batch_size: int = 32,
    target_update_freq: int = 100,
) -> Dict[str, List[float]]:
    """Train a DQN agent and return episode-level learning metrics."""
    replay_buffer = ExperienceReplay(capacity=10000)
    agent.update_target_network(target_agent)

    episode_rewards = []
    episode_costs = []
    episode_peak_demands = []
    epsilon_history = []
    loss_history = []

    print(f"Training DQN agent for {num_episodes} episodes...")
    for episode in range(num_episodes):
        state = env.reset()
        total_reward = 0.0
        episode_loss = 0.0
        loss_count = 0
        info = {}

        while True:
            action = agent.act(state, training=True)
            next_state, reward, done, info = env.step(action)
            replay_buffer.push(state, action, reward, next_state, done)

            if len(replay_buffer) > batch_size:
                batch = replay_buffer.sample(batch_size)
                for experience in batch:
                    batch_state, batch_action, batch_reward, batch_next_state, batch_done = experience
                    loss = agent.train_step(
                        batch_state,
                        batch_action,
                        batch_reward,
                        batch_next_state,
                        batch_done,
                        target_agent,
                    )
                    episode_loss += loss
                    loss_count += 1

            state = next_state
            total_reward += reward
            if done:
                break

        if (episode + 1) % target_update_freq == 0:
            agent.update_target_network(target_agent)

        episode_rewards.append(total_reward)
        episode_costs.append(info["episode_cost"])
        episode_peak_demands.append(info["peak_demand"])
        epsilon_history.append(agent.epsilon)
        loss_history.append(episode_loss / loss_count if loss_count else 0.0)

        if (episode + 1) % 100 == 0:
            recent_slice = slice(-100, None)
            average_reward = np.mean(episode_rewards[recent_slice])
            average_cost = np.mean(episode_costs[recent_slice])
            print(
                f"Episode {episode + 1}: "
                f"Mean reward = {average_reward:.2f}, "
                f"mean cost = ${average_cost:,.2f}, "
                f"epsilon = {agent.epsilon:.3f}"
            )

    print("Training completed.")
    return {
        "episode_rewards": episode_rewards,
        "episode_costs": episode_costs,
        "episode_peak_demands": episode_peak_demands,
        "epsilon_history": epsilon_history,
        "loss_history": loss_history,
    }


start_time = time.time()
training_metrics = train_dqn_agent(
    env,
    dqn_agent,
    target_dqn,
    num_episodes=2000,
    batch_size=32,
)
training_time = time.time() - start_time

print(f"Training time: {training_time:.2f} seconds")
print(f"Final epsilon: {dqn_agent.epsilon:.4f}")

## 6. Evaluate the learned policy

During evaluation, epsilon-greedy exploration is disabled: the agent selects the action with the largest predicted Q-value. Evaluation measures complete episodes and reports the mean and standard deviation of reward, cost, and peak demand. A random policy is evaluated separately as a simple reference. These results depend on the trained network and sampled episodes, so they should be interpreted from the output produced by running this notebook.

In [ ]:
def evaluate_agent(
    env: RLEnvironment,
    agent: DQN,
    num_episodes: int = 100,
) -> Dict[str, Any]:
    """Evaluate a greedy policy over independently reset episodes."""
    evaluation_rewards = []
    evaluation_costs = []
    evaluation_peak_demands = []

    print(f"Evaluating DQN agent for {num_episodes} episodes...")
    for _ in range(num_episodes):
        state = env.reset()
        total_reward = 0.0

        while True:
            action = agent.act(state, training=False)
            next_state, reward, done, info = env.step(action)
            state = next_state
            total_reward += reward
            if done:
                evaluation_rewards.append(total_reward)
                evaluation_costs.append(info["episode_cost"])
                evaluation_peak_demands.append(info["peak_demand"])
                break

    return {
        "mean_reward": float(np.mean(evaluation_rewards)),
        "std_reward": float(np.std(evaluation_rewards)),
        "mean_cost": float(np.mean(evaluation_costs)),
        "std_cost": float(np.std(evaluation_costs)),
        "mean_peak_demand": float(np.mean(evaluation_peak_demands)),
        "std_peak_demand": float(np.std(evaluation_peak_demands)),
        "all_rewards": evaluation_rewards,
        "all_costs": evaluation_costs,
        "all_peak_demands": evaluation_peak_demands,
    }


def evaluate_random_policy(
    env: RLEnvironment,
    num_episodes: int = 50,
) -> Dict[str, Any]:
    """Evaluate uniformly random actions as a reference policy."""
    random_rewards = []
    random_costs = []
    random_peak_demands = []

    for _ in range(num_episodes):
        state = env.reset()
        total_reward = 0.0

        while True:
            action = random.randrange(env.get_action_space_size())
            next_state, reward, done, info = env.step(action)
            state = next_state
            total_reward += reward
            if done:
                random_rewards.append(total_reward)
                random_costs.append(info["episode_cost"])
                random_peak_demands.append(info["peak_demand"])
                break

    return {
        "mean_reward": float(np.mean(random_rewards)),
        "mean_cost": float(np.mean(random_costs)),
        "mean_peak_demand": float(np.mean(random_peak_demands)),
        "all_rewards": random_rewards,
        "all_costs": random_costs,
        "all_peak_demands": random_peak_demands,
    }


evaluation_results = evaluate_agent(env, dqn_agent, num_episodes=100)
random_results = evaluate_random_policy(env, num_episodes=50)

print("\nDQN evaluation")
print(
    f"Mean reward: {evaluation_results['mean_reward']:.2f} "
    f"± {evaluation_results['std_reward']:.2f}"
)
print(
    f"Mean cost: ${evaluation_results['mean_cost']:,.2f} "
    f"± ${evaluation_results['std_cost']:,.2f}"
)
print(
    f"Mean peak demand: {evaluation_results['mean_peak_demand']:.2f} "
    f"± {evaluation_results['std_peak_demand']:.2f} MW"
)

print("\nRandom-policy reference")
print(f"Mean reward: {random_results['mean_reward']:.2f}")
print(f"Mean cost: ${random_results['mean_cost']:,.2f}")
print(f"Mean peak demand: {random_results['mean_peak_demand']:.2f} MW")

if not np.isclose(random_results["mean_cost"], 0.0):
    cost_reduction = (
        (random_results["mean_cost"] - evaluation_results["mean_cost"])
        / abs(random_results["mean_cost"])
        * 100.0
    )
    print(f"Cost reduction relative to the random policy: {cost_reduction:.1f}%")

## 7. Visualize training and evaluation

A moving average reduces the visual effect of episode-to-episode variation while retaining the underlying training record. The epsilon and loss panel shows the exploration schedule alongside the training loss; the two vertical axes have different units. The final panel compares mean cost and mean reward for the two evaluated policies. A visual trend is evidence about this run, not a proof of convergence or optimality.

In [ ]:
def create_dqn_visualizations(
    training_metrics: Dict[str, List[float]],
    evaluation_results: Dict[str, Any],
    random_results: Dict[str, Any],
) -> None:
    """Plot training histories and evaluation summaries."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("DQN Energy-Management Results", fontsize=16, fontweight="bold")

    episodes = np.arange(1, len(training_metrics["episode_rewards"]) + 1)
    window_size = 100

    reward_average = [
        np.mean(training_metrics["episode_rewards"][max(0, i - window_size + 1):i + 1])
        for i in range(len(episodes))
    ]
    axes[0, 0].plot(episodes, reward_average, color="tab:blue", label="100-episode moving average")
    axes[0, 0].set_xlabel("Episode")
    axes[0, 0].set_ylabel("Cumulative reward (dollars, with shaping terms)")
    axes[0, 0].set_title("Training reward")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    cost_average = [
        np.mean(training_metrics["episode_costs"][max(0, i - window_size + 1):i + 1])
        for i in range(len(episodes))
    ]
    axes[0, 1].plot(episodes, cost_average, color="tab:red", label="100-episode moving average")
    axes[0, 1].set_xlabel("Episode")
    axes[0, 1].set_ylabel("Episode cost ($)")
    axes[0, 1].set_title("Training cost")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    epsilon_axis = axes[1, 0]
    loss_axis = epsilon_axis.twinx()
    epsilon_line = epsilon_axis.plot(
        episodes,
        training_metrics["epsilon_history"],
        color="tab:green",
        label="Epsilon",
    )
    loss_episodes = episodes[::10]
    loss_values = training_metrics["loss_history"][::10]
    loss_line = loss_axis.plot(
        loss_episodes,
        loss_values,
        color="tab:orange",
        alpha=0.8,
        label="Mean episode training loss (sampled every 10 episodes)",
    )
    epsilon_axis.set_xlabel("Episode")
    epsilon_axis.set_ylabel("Exploration rate (epsilon)", color="tab:green")
    loss_axis.set_ylabel("Mean squared-error loss", color="tab:orange")
    epsilon_axis.set_title("Exploration and training loss")
    epsilon_axis.legend(epsilon_line + loss_line, [line.get_label() for line in epsilon_line + loss_line])
    epsilon_axis.grid(True, alpha=0.3)

    comparison_axis = axes[1, 1]
    methods = ["Random policy", "DQN agent"]
    costs = [random_results["mean_cost"], evaluation_results["mean_cost"]]
    rewards = [random_results["mean_reward"], evaluation_results["mean_reward"]]
    positions = np.arange(len(methods))
    bar_width = 0.35

    cost_bars = comparison_axis.bar(
        positions - bar_width / 2,
        costs,
        bar_width,
        label="Mean cost ($)",
        color="tab:red",
        alpha=0.7,
    )
    reward_axis = comparison_axis.twinx()
    reward_bars = reward_axis.bar(
        positions + bar_width / 2,
        rewards,
        bar_width,
        label="Mean reward",
        color="tab:blue",
        alpha=0.7,
    )
    comparison_axis.set_xlabel("Policy")
    comparison_axis.set_ylabel("Mean episode cost ($)", color="tab:red")
    reward_axis.set_ylabel("Mean episode reward", color="tab:blue")
    comparison_axis.set_xticks(positions)
    comparison_axis.set_xticklabels(methods)
    comparison_axis.set_title("Evaluation comparison")
    comparison_axis.legend(
        cost_bars + reward_bars,
        [bar.get_label() for bar in cost_bars + reward_bars],
        loc="best",
    )
    for bar, cost in zip(cost_bars, costs):
        comparison_axis.annotate(
            f"${cost:,.0f}",
            (bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
        )

    plt.tight_layout()
    plt.show()


create_dqn_visualizations(training_metrics, evaluation_results, random_results)

## 8. Inspect the learned policy

Action frequency shows which encoded choices the greedy policy selected during sampled episodes. It is descriptive rather than causal: a frequently selected action is not necessarily beneficial in every state. The state-action samples also show the observed price and predicted Q-value range, which can help reveal whether the network distinguishes actions. Q-values are learned estimates, not guaranteed costs or probabilities.

In [ ]:
def analyze_learned_policy(
    env: RLEnvironment,
    agent: DQN,
    num_test_episodes: int = 20,
) -> Tuple[Dict[int, int], Dict[int, List[float]], List[Dict[str, Any]]]:
    """Summarize greedy action use and selected state-action observations."""
    action_counts = {action: 0 for action in range(env.get_action_space_size())}
    action_rewards = {action: [] for action in range(env.get_action_space_size())}
    state_action_pairs = []

    for _ in range(num_test_episodes):
        state = env.reset()
        while True:
            q_values = agent.predict(state)
            action = int(np.argmax(q_values))
            decision_hour = min(env.current_hour, 24)
            decision_price = env.current_prices[decision_hour]

            next_state, reward, done, _ = env.step(action)
            action_counts[action] += 1
            action_rewards[action].append(reward)

            if len(state_action_pairs) < 100 and random.random() < 0.1:
                state_action_pairs.append({
                    "state": state.copy(),
                    "action": action,
                    "q_values": q_values.copy(),
                    "reward": reward,
                    "hour": decision_hour,
                    "price": decision_price,
                })

            state = next_state
            if done:
                break

    print("\nMost frequently selected actions")
    sorted_actions = sorted(action_counts.items(), key=lambda item: item[1], reverse=True)
    for rank, (action, count) in enumerate(sorted_actions[:10], start=1):
        if count:
            mean_reward = float(np.mean(action_rewards[action]))
            print(
                f"{rank:2d}. Action {action:2d}: {count:4d} selections, "
                f"mean immediate reward {mean_reward:9.2f}, "
                f"request {env.decode_action(action)}"
            )

    peak_hour_actions = []
    off_peak_actions = []
    for pair in state_action_pairs:
        if pair["price"] > 0.10:
            peak_hour_actions.append(pair["action"])
        else:
            off_peak_actions.append(pair["action"])

    def report_common_action(label: str, actions: List[int]) -> None:
        if actions:
            counts = {}
            for action in actions:
                counts[action] = counts.get(action, 0) + 1
            most_common = max(counts.items(), key=lambda item: item[1])[0]
            print(
                f"Most common sampled {label} action: {most_common} "
                f"({env.decode_action(most_common)})"
            )
        else:
            print(f"No sampled {label} decisions were available.")

    print("\nDecision patterns in the stored sample")
    report_common_action("higher-price", peak_hour_actions)
    report_common_action("lower-price", off_peak_actions)

    print("\nSample Q-value ranges")
    for sample_index, pair in enumerate(state_action_pairs[:5], start=1):
        minimum_q = float(np.min(pair["q_values"]))
        maximum_q = float(np.max(pair["q_values"]))
        print(
            f"Sample {sample_index}: hour {pair['hour']}, "
            f"price ${pair['price']:.2f}/kWh"
        )
        print(
            f"  Q range: [{minimum_q:.2f}, {maximum_q:.2f}]; "
            f"selected action: {pair['action']} "
            f"({env.decode_action(pair['action'])})"
        )

    return action_counts, action_rewards, state_action_pairs


action_counts, action_rewards, state_action_pairs = analyze_learned_policy(
    env,
    dqn_agent,
    num_test_episodes=20,
)

## 9. Interpreting the results and limitations

Read the plots and printed summaries together:

- A rising reward moving average can indicate that the agent is finding actions with higher shaped reward. Because reward includes peak penalties and a discharge bonus, check episode cost and peak demand as well rather than treating reward as a direct cost measure.
- A falling cost moving average is evidence of lower simulated energy-plus-demand cost during training episodes. Evaluation statistics show how the greedy policy performed on the evaluation episodes; the standard deviation describes variation across those episodes, not uncertainty about every possible real-world day.
- Epsilon describes the probability of a random training action. Training loss measures agreement with bootstrapped targets; low loss alone does not establish a good policy or convergence.
- The random policy is a reference, not an optimization benchmark. A cost reduction relative to it is specific to these environment rules, episode samples, and training run.
- Action frequencies and Q-values describe the network's decisions. They do not establish that a selected action is globally optimal or that the agent has learned a causal rule about electricity prices.

This is a compact teaching simulation, not a deployment-ready energy controller. It omits several operational constraints and uncertain influences, uses a partially observed representation of randomized prices, and provides no optimality guarantee. The natural scope of further study is to enrich the modeled observations and operational constraints while retaining a clearly specified environment and evaluating the resulting policy on appropriate scenarios. Such extensions are not implemented in this notebook.